# Занятие 6. Практика: кастомная трансформация и объединение данных

На прошлом занятии мы разобрали теорию: **работа со столбцами и строками**,
пользовательские функции через **`.apply()`** и **`lambda`**, строковый аксессор **`.str`**,
объединение таблиц через **`pd.concat()`** и **`pd.merge()`** (типы соединений `inner`,
`left`, `right`, `outer`).

Сегодня применяем всё это на реальных данных: парсим сырые текстовые поля (ФИО, почта,
телефон), строим новые признаки через `.apply(axis=1)` и собираем финальную таблицу
из основного датасета, второй волны учеников и трёх справочников.

Данные **синтетические** и не соответствуют действительности — телефоны, почты, ФИО, баллы,
города сгенерированы для учебных целей. Совпадения с реальными людьми случайны.

**План занятия:**
1. **Знакомство с данными** — что за пять таблиц лежит в `data/`;
2. **Разминка** — короткие примеры на `.apply`, `.str`, `concat`, `merge`;
3. **Практика** — 12 заданий на 30 баллов.


In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 220)
pd.set_option('display.precision', 2)

---
## Часть 1. Знакомство с данными

Сегодня у нас **пять файлов**:

| Файл | Что это |
|---|---|
| `ege_students.csv`       | основной датасет (то же, что раньше, но с 3 новыми колонками) |
| `ege_students_extra.csv` | «вторая волна» — 50 новых учеников для склейки через `concat` |
| `schools_by_city.csv`    | справочник городов |
| `subject_info.csv`       | справочник школьных предметов |
| `email_domains.csv`      | справочник почтовых доменов |

Загрузим всё и посмотрим по очереди.

In [ ]:
df = pd.read_csv('data/ege_students.csv', sep=';')
df_extra = pd.read_csv('data/ege_students_extra.csv', sep=';')
cities = pd.read_csv('data/schools_by_city.csv', sep=';')
subjects = pd.read_csv('data/subject_info.csv', sep=';')
domains = pd.read_csv('data/email_domains.csv', sep=';')

print('df:', df.shape)
print('df_extra:', df_extra.shape)
print('cities:', cities.shape)
print('subjects:', subjects.shape)
print('domains:', domains.shape)

### 1.1. Основной датасет — что нового

К уже знакомым колонкам добавились **три «сырых» текстовых поля** из формы регистрации.

In [ ]:
df.head()

In [ ]:
df.columns

**Новые колонки:**
- `ФИО_полное` — «Фамилия Имя Отчество» одной строкой;
- `email` — почта в формате `логин@домен`;
- `телефон` — номер, но в **разных форматах**.

Посмотрим на них крупным планом:

In [ ]:
df[['ФИО_полное', 'email', 'телефон']].head(10)

### 1.2. Телефоны — разнобой в форматах

Посмотрим на несколько случайных телефонов — как видно, они записаны в разных вариантах.

In [ ]:
df['телефон'].sample(15, random_state=0).tolist()

Форматов **пять**: `+7 (999) 123-45-67`, `89991234567`, `+7-999-123-45-67`,
`8 999 123 45 67`, `+79991234567`. В таком виде считать по ним ничего невозможно —
в практике вы напишете свою функцию нормализации.

### 1.3. Домены почт

Посмотрим, какие домены встречаются в колонке `email`:

In [ ]:
df['email'].head(10)

### 1.4. `df_extra` — дополнительные ученики

Формат тот же, что и в основном датасете. ID начинаются с **1001**, чтобы не пересекаться.

In [ ]:
df_extra.head()

In [ ]:
# сравним колонки — должны совпадать
print('одинаковые колонки?', df.columns.tolist() == df_extra.columns.tolist())

**Важное отличие:** в `df_extra` встречается город **«Владивосток»**, которого нет в основном датасете.

### 1.5. Справочник городов

In [ ]:
cities

**Внимание:** Владивостока здесь **нет**. Значит, после `merge` с этим справочником у учеников из Владивостока `федеральный_округ` и `рейтинг_школ` будут `NaN`.

### 1.6. Справочник предметов

In [ ]:
subjects

Тут покрыты **все** предметы, которые встречаются в датасете. Пропусков после merge не будет.

### 1.7. Справочник почтовых доменов

In [ ]:
domains

**Внимание:** в датасете есть университетский домен `edu.ru`, а в справочнике его **нет**. Для таких учеников поля `провайдер` и `страна` после merge будут `NaN`.

### 1.8. Итог знакомства

Держите в голове три места, где после объединения появятся `NaN`:

| Справочник | Что не покрыто |
|---|---|
| `cities`   | Владивосток |
| `subjects` | всё покрыто |
| `domains`  | `edu.ru` |

Это НЕ ошибка данных.

---
## Часть 2. Мини-разминка — вспоминаем приёмы

Каждый приём показан **на одном коротком примере, не связанном с задачами практики**.
Дальше приём применяете сами.

### 2.1. `.apply()` с обычной функцией

In [ ]:
def age_bucket(x):
    if x < 17:  return 'младше'
    if x == 17: return '17'
    return 'старше'

df['возраст'].apply(age_bucket)

### 2.2. `lambda` вместо `def`

In [ ]:
# признак «сдал ли на 90+» через lambda
df['балл_егэ'].apply(lambda x: x >= 90).sum()

### 2.3. `.str.split()` — деление строки

In [ ]:
# 'Нижний Новгород' → ['Нижний', 'Новгород']
df['город'].str.split(' ').head()

In [ ]:
# берём первое слово
df['город'].str.split(' ').apply(lambda x: x[0]).head()

### 2.4. `.apply(axis=1)` — функция по всей строке

In [ ]:
# часов подготовки на 1 час сна
df.apply(lambda row: row['часов_подготовки_в_неделю'] / (row['часов_сна'] + 1), axis=1).head()

### 2.5. `pd.concat` — склейка по строкам

In [ ]:
a = df.iloc[:3]
b = df.iloc[-3:]
pd.concat([a, b], ignore_index=True)

### 2.6. `pd.merge` — присоединяем справочник по ключу

In [ ]:
mini = pd.DataFrame({
    'тип_школы': ['Лицей', 'Гимназия', 'Обычная'],
    'престиж':   [10, 9, 5],
})
df[['имя', 'тип_школы']].head().merge(mini, on='тип_школы', how='left')

Инструменты вспомнили. **Дальше — практика.**

---
## Практика

Ниже — **12 заданий** на изученный материал. В каждом задании:
- **напишите код** в ячейке `# Ваш код` — код обязателен;
- **выведите результат** через `print()` или последней строкой.

В некоторых заданиях есть пункт **Вывод:** — там нужно короткое предложение о том, что получилось и как это трактовать.

Работаем с таблицами, загруженными выше: основной `df`, вторая волна `df_extra` и три справочника `cities`, `subjects`, `domains`.

**Максимум за практику — 30 баллов.**

### <font color='DarkOrange'>Задание 1 [2 балла]</font>

Из колонки `email` выделите **домен** (часть после `@`) и запишите её в новый столбец `домен_почты`.

Сколько учеников пользуются почтой на `yandex.ru`?

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 2 [2 балла]</font>

Из колонки `email` выделите **логин** (часть до `@`) и запишите в столбец `логин`.

Сколько логинов состоят **из 8 символов или меньше**?

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 3 [4 балла]</font>

Напишите функцию `normalize_phone(s)`, которая приводит любой формат телефона
(`+7 (999) 123-45-67`, `89991234567`, `+7-999-...`, `8 999 123 45 67`, `+79991234567`)
к единому виду `+7XXXXXXXXXX` (плюс, семёрка, 10 цифр — всего 12 символов).

Идея: оставить только цифры, если первая цифра `8` — заменить на `7`, потом добавить `+` в начало.

Примените функцию к колонке `телефон` и сохраните в `телефон_норм`.

У скольких учеников **нормализованный** телефон **отличается от исходного** — то есть строка в колонке `телефон` не совпадает со строкой в колонке `телефон_норм`?

> Смысл: посчитать, сколько записей реально «пострадали» от нормализации. Те номера, что и так были в формате `+7XXXXXXXXXX` без пробелов и скобок, останутся без изменений — их считать не нужно.

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 4 [2 балла]</font>

По колонке `телефон_норм` возьмите **первые 4 символа** — это «код региона» (пример: `+799`, `+791`).

Сколько учеников имеют телефон, начинающийся с `+799`?

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 5 [2 балла]</font>

Из колонки `ФИО_полное` (формат «Фамилия Имя Отчество») выделите **фамилию** —
первое слово — и сохраните в столбец `фамилия`.

Сколько учеников имеют фамилию, оканчивающуюся на `ов` или `ова`?

> Подсказка: `.str.endswith((...))` умеет принимать кортеж окончаний.

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 6 [3 балла]</font>

Создайте столбец `эффективность` по формуле:

$$
\text{эффективность} = \frac{\text{балл ЕГЭ}}{\text{часов подготовки в неделю} + 1}
$$

(единица в знаменателе — чтобы не делить на ноль). Используйте `.apply(axis=1)`.

Сколько учеников имеют эффективность **строго больше 10**?

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 7 [3 балла]</font>

Напишите функцию `categorize(row)`, которая возвращает `'звезда'`, если **одновременно**
`балл_егэ >= 90` и `средний_балл_года >= 4.5`, иначе — `'обычный'`.

Примените через `.apply(axis=1)` и сохраните результат в столбец `статус`.

Сколько получилось «звёзд»?

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 8 [3 балла]</font>

Склейте `df` и `df_extra` через `pd.concat`, взяв только **исходные 19 колонок**
(тех, что есть в `df_extra` — без ваших `домен_почты`, `логин`, `телефон_норм`,
`фамилия`, `эффективность`, `статус`).

Индексы сбросьте. Результат сохраните в `df_all`.

Сколько в `df_all` учеников из городов, которых **не было** в исходном `df` (то есть эти города «пришли» только со второй волной)?

> Смысл: `concat` расширяет датасет — среди новых строк могут появиться новые категории. Нужно найти множество городов, которые есть в `df_extra`, но отсутствуют в `df`, и посчитать, сколько строк в `df_all` относятся к этим городам.
>
> Подсказка: `set(a) - set(b)` — элементы `a`, которых нет в `b`. Затем `.isin(...)` по колонке `город`.

In [ ]:
# Ваш код


**Вывод:** *одним предложением — какой город пришёл только со второй волной и сколько учеников он принёс.*

### <font color='DarkOrange'>Задание 9 [2 балла]</font>

Присоедините справочник `cities` к `df_all` через `merge` по колонке `город`.
Тип соединения — такой, чтобы **никого не потерять** из `df_all`.
Результат снова сохраните в `df_all`.

У скольких учеников после merge оказался `NaN` в колонке `федеральный_округ`?

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 10 [2 балла]</font>

Из `df_all` возьмите учеников, чей `федеральный_округ == 'ПФО'`.

Какой у них **средний балл ЕГЭ**? Округлите до **двух знаков** после запятой.

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 11 [2 балла]</font>

Присоедините справочник `subjects` к `df_all` через `merge` по колонке `любимый_предмет`.
Тип соединения — снова такой, чтобы никого не потерять. Результат — в `df_all`.

Сколько в `df_all` учеников с `тип_предмета == 'гуманитарный'`?

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 12 [3 балла]</font>

Сначала создайте в `df_all` столбец `домен_почты` (мы это делали для `df`,
но `df_all` — другая таблица). Потом присоедините справочник `domains` через
`merge` на этот столбец (тип соединения — чтобы никого не потерять).

Сколько учеников удовлетворяют **одновременно** двум условиям:
- `страна == 'Россия'` (из справочника доменов),
- `балл_егэ >= 80`?

In [ ]:
# Ваш код


**Вывод:** *одним предложением — что даёт домен почты как признак и почему учеников на `edu.ru` (не Россия по справочнику) в подсчёт не попадает.*